In [1]:
import sys
sys.path.insert(0, '../')
sys.path.insert(1, '../../')
from build_config import ALL_SEASONS, COMPETITIONS, CURRENT_SEASON, INPUT_CSV_PATHS, TARGET_RANGES
from const import REPO_PATH, RAW_DATA_PATH, PROCESSED_DATA_PATH, MODELS_PATH
from fbref_const import URLs, TARGET_COLUMNS

from src.data.match_data_processing import process_match_target_var, process_match_other_var
from src.feature.feature_encoders import TeamEncoder, TeamLagFeatureGenerator, PreviousSeasonTeamAverager, TeamRestDaysCalculator, TeamLagTargetFeature

import pandas as pd
import os
import numpy as np
import mlflow
import joblib
import tempfile
import os

from build_utils import *

In [2]:
key_columns=['date', 'home', 'away']

In [3]:
team_encoder_path = f"{MODELS_PATH}/data_processors/{COMPETITIONS[0]}/team_encoder.pkl"
encoder = TeamEncoder.load(team_encoder_path)

In [4]:
seasons=sorted(ALL_SEASONS)

In [5]:
test_dict=dict((k, pd.read_csv(f'{REPO_PATH}/{INPUT_CSV_PATHS[k]}')) for k in COMPETITIONS)

In [6]:
all_features_dict={}
for competition in COMPETITIONS:
    if competition not in test_dict:
        continue
    test_df = test_dict[competition]
    team_encoder_path = f"{MODELS_PATH}/data_processors/{competition}/team_encoder.pkl"
    season_dfs_dict = {season: pd.read_csv(f"{PROCESSED_DATA_PATH}/{competition}/{season}/all_data_df.csv") for season in seasons}
    target_dfs = [pd.read_csv(f"{PROCESSED_DATA_PATH}/{competition}/{season}/all_target_df.csv") for season in seasons]
    # TeamEncoder features
    encoder = TeamEncoder.load(team_encoder_path)
    team_encoder_features = encoder.transform_spot(season_dfs_dict, test_df).reset_index(drop=True)

    # TeamLagFeatureGenerator features
    lag_feature_generator = TeamLagFeatureGenerator(lookback=5)
    team_lag_features = lag_feature_generator.transform_spot(season_dfs_dict, test_df).reset_index(drop=True)

    # PreviousSeasonTeamAverager features
    prev_season_averager = PreviousSeasonTeamAverager(decay_factor=1, date_col='date', home_col='home', away_col='away')
    prev_season_features = prev_season_averager.transform_spot(season_dfs_dict, test_df).reset_index(drop=True)

    # TeamRestDaysCalculator features
    rest_days_calculator = TeamRestDaysCalculator()
    rest_days_features = rest_days_calculator.transform_spot(season_dfs_dict, test_df).reset_index(drop=True)

    # TeamLagTargetFeature features
    lag_target_generator = TeamLagTargetFeature(lookback=5)
    lag_target_features = lag_target_generator.transform_spot(season_dfs_dict, target_dfs, test_df).reset_index(drop=True)

    # Merge all features on home, away, date
    from functools import reduce
    feature_dfs = [team_encoder_features, team_lag_features, prev_season_features, rest_days_features, lag_target_features]
    all_features = reduce(lambda left, right: pd.merge(left, right, on=['home', 'away', 'date'], how='outer'), feature_dfs)
    
    scaler = joblib.load(os.path.join(f"{MODELS_PATH}/data_processors/{competition}", 'standard_scaler.pkl'))
    all_features[scaler.feature_names_in_] = scaler.transform(all_features[scaler.feature_names_in_])

    all_features_dict[competition] = all_features.copy().fillna(-1)

In [7]:
train_features=pd.read_csv(f"{REPO_PATH}/data/features/premier_league/all_combined_features_2017-24.csv")
train_features['date'] = pd.to_datetime(train_features['date'])
train_features = train_features.merge(all_features_dict['premier_league'][['home', 'away', 'date']], on=['home', 'away', 'date'], how='right').fillna(-1)

for competition, df in all_features_dict.items():
    all_features_dict[competition] = df[train_features.columns]

In [8]:
# Compare train_features and all_features_dict['premier_league'] (excluding home, away, date)
tf_values = train_features.drop(columns=['home', 'away', 'date']).values
af_values = all_features_dict['premier_league'].drop(columns=['home', 'away', 'date']).values

# Find where they differ
diff_mask = 1-np.isclose(tf_values, af_values)
diff_indices = np.argwhere(diff_mask)
print(f"Number of differing values: {diff_indices.shape[0]}")
print("First 10 differences:")
train_feature_cols = train_features.drop(columns=['home', 'away', 'date']).columns.tolist()
all_feature_cols = all_features_dict['premier_league'].drop(columns=['home', 'away', 'date']).columns.tolist()
for idx, (row, col) in enumerate(diff_indices[:10]):
    home = train_features.iloc[row]['home']
    away = train_features.iloc[row]['away']
    date = train_features.iloc[row]['date']
    tf_val = tf_values[row, col]
    af_val = af_values[row, col]
    col_name = train_feature_cols[col]
    col_name_af = all_feature_cols[col]

    print(f"Row {row}, Column '{col_name}, {col_name_af}': home={home}, away={away}, date={date}, train_features={tf_val}, all_features_dict={af_val}")

Number of differing values: 0
First 10 differences:


In [9]:
model_dict={
    'premier_league': {
        'away_goals':{
            'run_id': 'f2fcfc710a2740ff859f60ca0a1981c9',
            'artifact_path': 'model',
        },
        'home_goals':{
            'run_id': '143e604214444d8f90e88b6e47693328',
            'artifact_path': 'model',
        },
    }
}

In [10]:
def load_joblib_model_from_mlflow(run_id, artifact_path, tracking_uri=None):
    """
    Fetch and load a joblib-dumped model from MLflow given experiment_id, run_id, and artifact_path.
    Optionally specify the MLflow tracking URI.
    Returns the loaded model.
    """
    if tracking_uri is not None:
        mlflow.set_tracking_uri(tracking_uri)
    client = mlflow.tracking.MlflowClient()
    # Download artifact to a temporary directory
    with tempfile.TemporaryDirectory() as tmp_dir:
        local_path = client.download_artifacts(run_id, artifact_path, tmp_dir)
        # Find the first .joblib file in the artifact directory
        for root, _, files in os.walk(local_path):
            for file in files:
                if file.endswith('.joblib'):
                    model_path = os.path.join(root, file)
                    return joblib.load(model_path)
        raise FileNotFoundError("No .joblib model file found in the artifact path.")


In [11]:
predictions={}

In [12]:
for competition in COMPETITIONS:
    predictions[competition] = {}
    for target_name in TARGET_RANGES:
        model = load_joblib_model_from_mlflow(model_dict[competition][target_name]['run_id'],
                                                model_dict[competition][target_name]['artifact_path'],
                                                f'{REPO_PATH}/mlflow')
        prediction = model.predict_proba(all_features_dict[competition].drop(columns=key_columns))
        prediction = pd.DataFrame(prediction, columns=list(range(TARGET_RANGES[target_name][0], TARGET_RANGES[target_name][1]+1))+['other'])
        predictions[competition][target_name]=prediction

/Users/tianqihuang/anaconda3/envs/betbot/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
# Add home, away, date columns from test_dict to each predictions DataFrame
for competition in predictions:
    test_rows = all_features_dict[competition][['home', 'away', 'date']].reset_index(drop=True)
    for target_name in predictions[competition]:
        df = predictions[competition][target_name]
        # Prepend home, away, date columns
        df = pd.concat([test_rows, df.reset_index(drop=True)], axis=1)
        predictions[competition][target_name] = df

# Example: predictions['premier_league']['home_goals'].head()

In [14]:
predictions['premier_league']['home_goals']

,home,away,date,0,1,2,3,4,5,other
0,Arsenal,Newcastle Utd,2020-02-16,0.000014,0.000022,0.000028,0.000015,0.999921,2.502847e-07,6.308530e-07
1,Aston Villa,Tottenham,2020-02-16,0.000009,0.000034,0.999937,0.000006,0.000005,5.538296e-06,3.399492e-06
2,Brighton,Watford,2020-02-08,0.000016,0.999920,0.000023,0.000011,0.000016,4.270537e-06,1.019807e-05
3,Chelsea,Manchester Utd,2020-02-17,0.999920,0.000026,0.000026,0.000009,0.000010,6.865392e-06,2.520598e-06
4,Everton,Crystal Palace,2020-02-08,0.000002,0.000006,0.000004,0.999976,0.000007,2.157559e-06,1.871797e-06
5,Manchester City,West Ham,2020-02-19,0.000005,0.000007,0.999967,0.000008,0.000006,4.024415e-06,3.536223e-06
6,Norwich City,Liverpool,2020-02-15,0.999974,0.000007,0.000007,0.000004,0.000003,2.599786e-06,1.495964e-06
7,Sheffield Utd,Bournemouth,2020-02-09,0.000004,0.000006,0.999978,0.000003,0.000005,1.943179e-06,2.533412e-06
8,Southampton,Burnley,2020-02-15,0.000013,0.999936,0.000023,0.000005,0.000008,5.070035e-06,1.043239e-05
9,Wolves,Leicester City,2020-02-14,0.999936,0.000018,0.000016,0.000010,0.000005,1.083370e-05,4.470779e-06


In [15]:
predictions['premier_league']['away_goals']

,home,away,date,0,1,2,3,4,5,6,other
0,Arsenal,Newcastle Utd,2020-02-16,9.999815e-01,0.000005,2.079341e-06,3.089164e-06,3.138954e-06,8.355187e-07,0.000002,0.000003
1,Aston Villa,Tottenham,2020-02-16,7.959486e-06,0.000002,1.657684e-05,9.999551e-01,7.191365e-06,5.759091e-06,0.000001,0.000004
2,Brighton,Watford,2020-02-08,6.841768e-06,0.999974,2.483416e-06,5.067849e-06,1.553210e-06,3.361596e-06,0.000003,0.000004
3,Chelsea,Manchester Utd,2020-02-17,1.544783e-05,0.000016,9.999425e-01,8.744177e-06,1.449186e-06,5.132211e-06,0.000005,0.000006
4,Everton,Crystal Palace,2020-02-08,2.444666e-06,0.999987,1.376587e-06,5.277982e-07,7.355339e-07,2.363786e-06,0.000003,0.000003
5,Manchester City,West Ham,2020-02-19,9.999723e-01,0.000008,3.746262e-06,3.336777e-06,2.409791e-06,1.657464e-06,0.000004,0.000005
6,Norwich City,Liverpool,2020-02-15,6.767376e-07,0.999990,9.329542e-07,2.168776e-06,5.911550e-07,1.275378e-06,0.000002,0.000002
7,Sheffield Utd,Bournemouth,2020-02-09,3.872475e-06,0.999988,6.695693e-07,1.447505e-06,9.841750e-07,6.934085e-07,0.000001,0.000002
8,Southampton,Burnley,2020-02-15,5.409830e-06,0.000017,9.999588e-01,7.236157e-06,1.425744e-06,2.939574e-06,0.000002,0.000005
9,Wolves,Leicester City,2020-02-14,9.999710e-01,0.000005,9.502585e-07,6.271933e-06,3.936350e-06,2.358426e-06,0.000006,0.000004


In [16]:
# Rename columns and update values in predictions DataFrames for each competition and target
for competition in predictions:
    for target_name in predictions[competition]:
        df = predictions[competition][target_name]
        cols = df.columns.tolist()
        # Only rename and update non-metadata columns (assume first three are home, away, date)
        meta_cols = ['home', 'away', 'date']
        feature_cols = cols[3:]
        # Rename columns: first becomes lte_{original}, others become gt_{previous}
        new_cols = meta_cols.copy()
        if feature_cols:
            new_cols.append(f"lte_{feature_cols[0]}")
            for i in range(1, len(feature_cols)):
                new_cols.append(f"gt_{feature_cols[i-1]}")
        # Update values: first feature column stays, others become sum of itself and all to the right
        arr = df[feature_cols].values.copy() if feature_cols else None
        if arr is not None and arr.shape[1] > 0:
            for i in range(1, arr.shape[1]):
                arr[:, i] = arr[:, i:].sum(axis=1)
            df = pd.concat([df[meta_cols].reset_index(drop=True), pd.DataFrame(arr, columns=new_cols[3:])], axis=1)
            df.columns = new_cols
            predictions[competition][target_name] = df

# Example: predictions['premier_league']['home_goals'].head()

In [17]:
predictions['premier_league']['home_goals']

,home,away,date,lte_0,gt_0,gt_1,gt_2,gt_3,gt_4,gt_5
0,Arsenal,Newcastle Utd,2020-02-16,0.000014,0.999986,0.999964,0.999936,0.999922,8.811377e-07,6.308530e-07
1,Aston Villa,Tottenham,2020-02-16,0.000009,0.999991,0.999957,0.000020,0.000014,8.937787e-06,3.399492e-06
2,Brighton,Watford,2020-02-08,0.000016,0.999984,0.000063,0.000041,0.000030,1.446860e-05,1.019807e-05
3,Chelsea,Manchester Utd,2020-02-17,0.999920,0.000080,0.000054,0.000028,0.000019,9.385990e-06,2.520598e-06
4,Everton,Crystal Palace,2020-02-08,0.000002,0.999998,0.999991,0.999987,0.000011,4.029356e-06,1.871797e-06
5,Manchester City,West Ham,2020-02-19,0.000005,0.999995,0.999989,0.000022,0.000014,7.560639e-06,3.536223e-06
6,Norwich City,Liverpool,2020-02-15,0.999974,0.000026,0.000019,0.000012,0.000007,4.095750e-06,1.495964e-06
7,Sheffield Utd,Bournemouth,2020-02-09,0.000004,0.999996,0.999991,0.000013,0.000009,4.476590e-06,2.533412e-06
8,Southampton,Burnley,2020-02-15,0.000013,0.999987,0.000051,0.000028,0.000023,1.550243e-05,1.043239e-05
9,Wolves,Leicester City,2020-02-14,0.999936,0.000064,0.000046,0.000030,0.000020,1.530447e-05,4.470779e-06


In [18]:
predictions['premier_league']['away_goals']

,home,away,date,lte_0,gt_0,gt_1,gt_2,gt_3,gt_4,gt_5,gt_6
0,Arsenal,Newcastle Utd,2020-02-16,9.999815e-01,0.000019,0.000014,0.000012,0.000009,0.000005,0.000005,0.000003
1,Aston Villa,Tottenham,2020-02-16,7.959486e-06,0.999992,0.999990,0.999973,0.000018,0.000011,0.000005,0.000004
2,Brighton,Watford,2020-02-08,6.841768e-06,0.999993,0.000019,0.000017,0.000012,0.000010,0.000007,0.000004
3,Chelsea,Manchester Utd,2020-02-17,1.544783e-05,0.999984,0.999969,0.000026,0.000017,0.000016,0.000011,0.000006
4,Everton,Crystal Palace,2020-02-08,2.444666e-06,0.999998,0.000011,0.000009,0.000009,0.000008,0.000006,0.000003
5,Manchester City,West Ham,2020-02-19,9.999723e-01,0.000028,0.000019,0.000015,0.000012,0.000010,0.000008,0.000005
6,Norwich City,Liverpool,2020-02-15,6.767376e-07,0.999999,0.000009,0.000008,0.000006,0.000005,0.000004,0.000002
7,Sheffield Utd,Bournemouth,2020-02-09,3.872475e-06,0.999996,0.000008,0.000007,0.000006,0.000005,0.000004,0.000002
8,Southampton,Burnley,2020-02-15,5.409830e-06,0.999995,0.999977,0.000019,0.000012,0.000010,0.000007,0.000005
9,Wolves,Leicester City,2020-02-14,9.999710e-01,0.000029,0.000024,0.000023,0.000016,0.000012,0.000010,0.000004
